## Task 1:


In [1]:
import pandas as pd

# to get the list of columns so i can copy and paste them to map them in load.py
df = pd.read_csv("Data_Engineering_Challenge.csv")
df.columns.tolist()

['Site Code',
 'Timestamp',
 'Source Tag',
 'Solar Output Current',
 'Total Load Current',
 'Battery Total Current',
 'Total Voltage']

In [2]:
# checking if lambda function works to strip all space from the columns that have them
df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

df.dtypes

Site Code                    str
Timestamp                    str
Source Tag                   str
Solar Output Current     float64
Total Load Current       float64
Battery Total Current    float64
Total Voltage            float64
dtype: object

In [3]:
# checking for wrong entries
Source_Tag = df["Source Tag"].unique()
print(Source_Tag)

<StringArray>
['Battery             ', 'DG                  ', 'DG+Solar            ',
 'Solar+Battery       ', 'DG+Solar+Battery    ', 'DG+Battery          ']
Length: 6, dtype: str


## Task 2:

Notes:

- The data have 3 min time interval, the challenge demands a 1 hour interval time series data which is aggregated and grouped

- We need to calculate runing hours of all power sources which is going to be:
  `(number of readings of a source in 1 hours * (3 / 60))`

- In many rows of Source Tag the sources are concatinated with a "+" sign So if the substring is present in the main string we have to treat is as "Active" Power Source. and we can use `str.contains()` func to check if the substring is present.

- Output Format is kind of confusing but this is just a dataframe of 1 hour of DG now all of it.


In [4]:
import pandas as pd
import load_data

# loading data using load_data.py
df = load_data.load_dataset("Data_Engineering_Challenge.csv")
df.head()

,Site Code,Timestamp,Source Tag,Solar Output Current,Total Load Current,Battery Total Current,Total Voltage
0,TEST-01,2026-05-09 19:00:00+00:00,Battery,0.0,115.3,-119.3,48.1
1,TEST-01,2026-05-09 19:03:00+00:00,Battery,0.0,115.7,-115.7,48.1
2,TEST-01,2026-05-09 19:06:00+00:00,DG,0.0,109.3,112.1,48.9
3,TEST-01,2026-05-09 19:09:00+00:00,DG,0.0,111.1,114.0,49.1
4,TEST-01,2026-05-09 19:12:00+00:00,DG,0.0,117.1,111.0,49.3


In [5]:
df["Hourly_Timestamp"] = df["Timestamp"].dt.floor("h")

df.head()

,Site Code,Timestamp,Source Tag,Solar Output Current,Total Load Current,Battery Total Current,Total Voltage,Hourly_Timestamp
0,TEST-01,2026-05-09 19:00:00+00:00,Battery,0.0,115.3,-119.3,48.1,2026-05-09 19:00:00+00:00
1,TEST-01,2026-05-09 19:03:00+00:00,Battery,0.0,115.7,-115.7,48.1,2026-05-09 19:00:00+00:00
2,TEST-01,2026-05-09 19:06:00+00:00,DG,0.0,109.3,112.1,48.9,2026-05-09 19:00:00+00:00
3,TEST-01,2026-05-09 19:09:00+00:00,DG,0.0,111.1,114.0,49.1,2026-05-09 19:00:00+00:00
4,TEST-01,2026-05-09 19:12:00+00:00,DG,0.0,117.1,111.0,49.3,2026-05-09 19:00:00+00:00


In [ ]:
def running_hours(series):
    source_tags = ["Battery", "DG", "Solar"]
    rows = []

    for source_tag in source_tags:
        reps = (
            series["Source Tag"].str.contains(source_tag, case=False).sum()
        )  # reps is how many times a source tag appear
        run_hours = (reps * (3 / 60)).round(2)

        # now we check if run_hours are greater than 0 and then append the dict to rows
        if run_hours > 0:
            rows.append(
                {
                    "site code": series["Site Code"].iloc[0],
                    "hour window": series.name,
                    "source": source_tag,
                    "run_hours": run_hours,
                }
            )

    return pd.DataFrame(rows)

In [ ]:
result = df.groupby("Hourly_Timestamp").apply(running_hours)
result.reset_index(drop=True, inplace=True)

site code                      str
hour window    datetime64[us, UTC]
source                         str
run_hours                  float64
dtype: object

In [ ]:
# writing to csv
result.to_csv("result.csv", index=False)